# CELIOS
### Sample - usage notebook for vis2024 dataset
Notebook to generate and visualize celios outputs

In [ ]:
# Import necessary modules
import pandas
import os
import sys
import shutil
import pandas
import plotly.graph_objects as go
from celios.core import run_celios

In [2]:
# This cell sets up the project root so the notebook can import project code and data

# Get the directory of the current notebook (notebooks/)
notebook_dir = os.getcwd()

# Go up one level to reach CELIOS/
ROOT = os.path.dirname(notebook_dir)

# Verify we found the right location
if not os.path.isdir(os.path.join(ROOT, 'data')):
    raise FileNotFoundError(f"Expected 'data' folder at {ROOT}, but it doesn't exist")

# Add to path if not already there
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print(f"Project ROOT: {ROOT}")
print(f"Data folder exists: {os.path.isdir(os.path.join(ROOT, 'data'))}")

Project ROOT: c:\Users\viviamsb\OneDrive - NTNU\PhD Folder\Pipeline\TRAFIKK\trafikk\Modules\celios
Data folder exists: True


## 1. LOAD INPUT

In [3]:
# Load cell line metadata

metadata = pandas.read_excel(os.path.join(ROOT, 'data', 'vis_2024', 'S1_cell_line_drug_info.xlsx'), engine='openpyxl')
tissue_types = metadata['tissue'].unique().tolist()
tissue_types.sort()
print("Available tissue types:")
for tissue in tissue_types:
    print(tissue)

Available tissue types:
Adrenal Gland
Biliary Tract
Bladder
Bone
Breast
Central Nervous System
Cervix
Endometrium
Esophagus
Haematopoietic and Lymphoid
Head and Neck
Kidney
Large Intestine
Liver
Lung
Ovary
Pancreas
Peripheral Nervous System
Prostate
Skin
Soft Tissue
Stomach
Testis
Thyroid
Uterus
Vulva


## 2. SELECT 1-TISSUE

In [4]:
# Select tissue type
selected_tissue = 'Head and Neck'  # Change this to select a different tissue type

In [5]:
# Filter metadata for selected tissue type
tissue_metadata = metadata[metadata['tissue'] == selected_tissue]
tissue_metadata = tissue_metadata[['model_name', 'model_id']].reset_index(drop=True)
tissue_metadata = tissue_metadata.rename(columns={'model_id': 'SIDM', 'model_name': 'cell_line_name'})
tissue_metadata.to_csv(os.path.join(ROOT, 'data', 'vis_2024', 'cell_line_list.csv'), index=False)
print(f"Shape of filtered metadata: {tissue_metadata.shape}")

# Clean list of cell line names (uppercase, remove spaces and special characters)
cell_names = tissue_metadata['cell_line_name'].tolist()
cleaned_cell_names = []
for name in cell_names:
    if isinstance(name, int):
        cleaned_cell_names.append(str(name))
    else:
        clean_name = ''.join(e for e in name if e.isalnum()).upper()
        cleaned_cell_names.append(clean_name)
# cleaned_cell_names

Shape of filtered metadata: (27, 2)


## 3. SELECT ALL TISSUES

In [17]:
all_tissues = metadata[['tissue', 'model_name', 'model_id']].drop_duplicates().reset_index(drop=True)
all_tissues = all_tissues.rename(columns={'tissue': 'Tissue', 'model_name': 'cell_line_name', 'model_id': 'SIDM'})

# Convert cell line names to string, uppercase, and keep spaces and special characters ('-')
def clean_cell_line_name(name):
    if isinstance(name, int):
        return str(name)
    else:
        return ''.join(e for e in name if e.isalnum() or e in [' ', '-', '+']).upper()
    
all_tissues['cell_line_name'] = all_tissues['cell_line_name'].apply(clean_cell_line_name)

# Map tissues to short names:
tissue_mapping = {
    'Haematopoietic and Lymphoid' : 'Lymphoid',
    'Head and Neck' : 'Head_neck',
    'Large Intestine' : 'Large_intestine',
    'Peripheral Nervous System': 'PNS',
    'Central Nervous System': 'CNS',
    'Biliary Tract': 'Biliary_tract',
    'Adrenal Gland': 'Adrenal',
    'Soft Tissue': 'Soft_tissue',
}

all_tissues['Tissue'] = all_tissues['Tissue'].replace(tissue_mapping)
all_tissues.to_csv(os.path.join(ROOT, 'data', 'vis_2024', 'all_cell_lines.csv'), index=False)

## 4. RUN CELIOS

In [26]:
# SELECTED TISSUE TYPE CONFIG

config_tissue = {
    "paths": {
        "base": ROOT,
        "input": os.path.join(ROOT, "data"),
        "output": os.path.join(ROOT, "data", "vis_2024", "results", selected_tissue),
        "cellfiles_dir": os.path.join(ROOT, "data", "vis_2024", "results", selected_tissue),
    },
    "steps": {
        "Node": {# relative to `paths.input`
            "node_input": os.path.join(ROOT, "data", "node_dic_input", "cell_fate_plus.sif"),
            "hgnc_symbols_file": os.path.join(ROOT, "data", "node_dic_input", "hgnc_complete_set.txt"),
            "manual_symbols_file": os.path.join(ROOT, "data", "node_dic_input", "manual_symbols.csv"),
            "include_alias_previous_symbols": False,
            "directory_output": os.path.join(ROOT, "data", "vis_2024", "results", selected_tissue),
        },
        "Activity": {# relative to `paths.input`
            "activity_file": os.path.join(ROOT, "data", "activity_input", "rnaseq_tpm_20220624.csv"),
            "cell_line_file": os.path.join(ROOT, "data", "vis_2024", "cell_line_list.csv"),
            "tf_activity_file": os.path.join(ROOT, "data", "activity_input", "ccle_tf_activities.csv"),
            "mutations_file": os.path.join(ROOT, "data", "activity_input", "CCLE_muts_binary.csv"),
            "cnv_file": os.path.join(ROOT, "data", "activity_input", "CCLE_CNV_binary.csv"),
            "directory_output": os.path.join(ROOT, "data", "vis_2024", "results", selected_tissue),
            "data_sources": ["mutations", "cnv", "TF"],
        },
    },
}

In [18]:
# ALL TISSUES CONFIG

config_all = {
    "paths": {
        "base": ROOT,
        "input": os.path.join(ROOT, "data"),
        "output": os.path.join(ROOT, "data", "vis_2024", "results"),
        "tissue_dir": os.path.join(ROOT, "data", "vis_2024", "results", "all_tissues"),
    },
    "steps": {
        "Node": {# relative to `paths.input`
            "node_input": os.path.join(ROOT, "data", "node_dic_input", "cell_fate_plus.sif"),
            "hgnc_symbols_file": os.path.join(ROOT, "data", "node_dic_input", "hgnc_complete_set.txt"),
            "manual_symbols_file": os.path.join(ROOT, "data", "node_dic_input", "manual_symbols.csv"),
            "include_alias_previous_symbols": False,
            "directory_output": os.path.join(ROOT, "data", "vis_2024", "results", "all_tissues"),            
        },
        "Activity": {# relative to `paths.input`
            "activity_file": os.path.join(ROOT, "data", "activity_input", "rnaseq_tpm_20220624.csv"),
            "cell_line_file": os.path.join(ROOT, "data", "vis_2024", "all_cell_lines.csv"),
            "tf_activity_file": os.path.join(ROOT, "data", "activity_input", "ccle_tf_activities.csv"),
            "mutations_file": os.path.join(ROOT, "data", "activity_input", "CCLE_muts_binary.csv"),
            "cnv_file": os.path.join(ROOT, "data", "activity_input", "CCLE_CNV_binary.csv"),
            "directory_output": os.path.join(ROOT, "data", "vis_2024", "results", "all_tissues"),
            "data_sources": ["mutations", "cnv", "TF"],
        },
    },
}

In [27]:
# SELECTED TISSUE TYPE RUN
celios_tissue_output = run_celios(config_tissue,
                           plan=False,
                           dna_damage_state=False,
                           verbose=True
                           )

STEP 1: Node dictionary
Ensured directory: c:\Users\viviamsb\OneDrive - NTNU\PhD Folder\Pipeline\CELIOS\data
Ensured directory: c:\Users\viviamsb\OneDrive - NTNU\PhD Folder\Pipeline\CELIOS\data\vis_2024\results\Head and Neck
Processing node list (in-memory) with 63 entries



STEP 2: Extracting omics - activity matrix


STEP 3: Writing DrugLogic pipeline training files
Using cell files directory: c:\Users\viviamsb\OneDrive - NTNU\PhD Folder\Pipeline\CELIOS\data\vis_2024\results\Head and Neck
Wrote 27 training files across 27 cell-line directories to: c:\Users\viviamsb\OneDrive - NTNU\PhD Folder\Pipeline\CELIOS\data\vis_2024\results\Head and Neck\cell_lines

CELIOS pipeline completed successfully.


In [19]:
# ALL TISSUES TYPE RUN
celios_all_output = run_celios(config_all,
                           plan=False,
                           dna_damage_state=False,
                           verbose=True
                           )

STEP 1: Node dictionary
Ensured directory: c:\Users\viviamsb\OneDrive - NTNU\PhD Folder\Pipeline\CELIOS\data
Ensured directory: c:\Users\viviamsb\OneDrive - NTNU\PhD Folder\Pipeline\CELIOS\data\vis_2024\results
Processing node list (in-memory) with 63 entries



STEP 2: Extracting omics - activity matrix


STEP 3: Writing DrugLogic pipeline training files (tissue-organized)
Using tissue directory: c:\Users\viviamsb\OneDrive - NTNU\PhD Folder\Pipeline\CELIOS\data\vis_2024\results\all_tissues
Wrote training files to 757 cell-line directories across 26 tissue folders

CELIOS pipeline completed successfully.


## 5. CHECK OUTPUTS

In [29]:
# Unpack CELIOS output
activity_matrix = celios_tissue_output['activity_matrix']

# Skip symbol column
activity_matrix = activity_matrix.drop(columns=['symbol'])

# Number of cell lines processed
num_cell_lines = activity_matrix.shape[1]
print(f"Number of cell lines processed: {num_cell_lines}")

# Number of nodes in the activity matrix
num_nodes = activity_matrix.shape[0]
print(f"Number of nodes in the activity matrix: {num_nodes}")

Number of cell lines processed: 27
Number of nodes in the activity matrix: 63


In [3]:
# Load master matrix for cell line inspection
master_matrix = pandas.read_csv(os.path.join(ROOT, 'data', 'vis_2024', 'results', 'all_tissues', 'activity_master_matrix.csv'))

# Select cell line in the master matrix
cell_line_to_inspect = 'SIDM00591' # Change this to the desired cell line ID

# Find all data for the selected cell line and organize by data type
cell_line_data = {'symbol': master_matrix['symbol']}

for col in master_matrix.columns:
    if col.startswith(cell_line_to_inspect + '_'):
        data_type = col.split('_', 1)[1]
        cell_line_data[data_type] = master_matrix[col]

# Build new dataframe with symbol and data types
cell_line_df = pandas.DataFrame(cell_line_data)
print(f"Columns in new dataframe: {cell_line_df.columns.tolist()}")
# cell_line_df

Columns in new dataframe: ['symbol', '_TF', '_mutations', '_cnv']


## 6. PLOT RESULTS

In [34]:
# HEATMAP OF ACTIVITY SCORES
fig = go.Figure(data=go.Heatmap(
    z=activity_matrix.values,
    x=activity_matrix.columns,
    y=activity_matrix.index,
    colorscale='Tropic',
    hoverongaps=True
))

fig.update_layout(
    title=f'Activity Scores Heatmap for {selected_tissue} Cell Lines',
    xaxis_nticks=num_cell_lines,
    yaxis_nticks=num_nodes,
    yaxis_title='Nodes',
    xaxis_title='Cell Lines',
    height=1200,
    width=1200
)
fig.show()

In [35]:
# HEATMAP OF A SINGLE CELL LINE ACROSS DATA TYPES
fig = go.Figure(data=go.Heatmap(
    z=cell_line_df.drop(columns=['symbol']).values,
    x=cell_line_df.drop(columns=['symbol']).columns,
    y=cell_line_df['symbol'],
    colorscale='Tropic',
    hoverongaps=True
))
fig.update_layout(
    title=f'Activity Scores Heatmap for Cell Line {cell_line_to_inspect}',
    yaxis_nticks=num_nodes,
    yaxis_title='Nodes',
    xaxis_title='Data Types',
    height=1200,
    width=600
)
fig.show()

## EXTRA: copy pipeline files

In [17]:
# COPY MAIN FILES TO CELL LINE FOLDERS 

def copy_files_to_cell_line_folders(general_files_dir, cell_line_directory):
    for folder_name in os.listdir(cell_line_directory):
        folder_path = os.path.join(cell_line_directory, folder_name)
        if os.path.isdir(folder_path):
            # Copy files from general_files_dir to the cell line folder
            for filename in os.listdir(general_files_dir):
                src_file = os.path.join(general_files_dir, filename)
                dest_file = os.path.join(folder_path, filename)
                shutil.copy(src_file, dest_file)

In [18]:
general_files_dir = os.path.join(ROOT, 'consensus', 'results', 'general_files')
cell_line_directory = os.path.join(ROOT, 'consensus', 'results', 'cell_lines')
copy_files_to_cell_line_folders(general_files_dir, cell_line_directory)